# Day 09 · 模块与高级特性

> **前置**: Day01-08 全部(重点:函数/OOP/文件 I/O)
> **关键问题**: 代码破千行后,单文件不堪重负 —— 怎么把代码**拆分到多个文件**、让重复的"计时/日志"逻辑自动复用?本节是工具箱补完课:包、生成器、上下文管理器、装饰器。

**引入:千行文件滚动条细如发丝**

抽问上节: Python 里 `class` 和 `__init__` 分别是什么?(蓝图 / 自动跑的构造函数)。再问: 一个 `utils.py` 有 2000 
行,你怎么管理?引出**模块化 + 包**;再抛一个需求:"给任何函数自动计时" —— 吊足胃口。


**1. `import` / `from` / `as` 三种导入**

口诀:**`import 模块` 带前缀,`from` 直接用,`as` 改别名**。把函数/类放进 `utils.py`,同目录 `import utils` 
即可。**包**(Package) = 模块的文件夹,必须含 `__init__.py`(可以为空)。


In [ ]:
import math
print(math.sqrt(16))            # 4.0,通过模块名访问

from math import sqrt
print(sqrt(16))                 # 4.0,直接用成员名

from math import factorial as fac
print(fac(5))                   # 120,别名 fac

包结构示例:创建 `mypkg/` 文件夹,里面放 `__init__.py`(空文件即可)和 `tools.py`。主程序 `from mypkg.tools import 函数名` 
即可使用。`__init__.py` 让 Python 把文件夹识别为"包",没有它就是个普通文件夹。


In [ ]:
# 包目录结构示例(伪代码,实际需创建文件)
# mypkg/
# ├── __init__.py      ← 必须有,可为空
# ├── tools.py         ← 放工具函数
# └── helpers.py       ← 放辅助函数
#
# 主程序:
# from mypkg.tools import greet
# greet()

**✏️ 练一练 ⭐⭐**

把 Day08 的 `Dog` 类移到独立文件 `dog.py`,然后在当前 notebook 中 `from dog import Dog` 实例化并调用 `bark()`。

In [ ]:
# ======================
# 学员代码区
# ======================


In [ ]:
# 参考答案(先在当前目录建 dog.py 再运行)
# dog.py 内容:
# class Dog:
#     def __init__(self, name):
#         self.name = name
#     def bark(self):
#         print(f"{self.name}: 汪汪!")

from dog import Dog
d = Dog("旺财")
d.bark()                        # 旺财: 汪汪!

**2. 生成器 `yield`**

普通函数 `return` 一次就结束,想**逐个产出**数据且**不占内存**,用 `yield`。生成器是惰性计算的迭代器。口诀:**`yield` 是 `return` 
的"Hold"版本 —— 产出值但不退场**,下次 `next()` 从这里恢复。


In [ ]:
def 平方(limit):
    for i in range(limit):
        yield i * i             # 每次产一个值,暂停

gen = 平方(4)
print(next(gen))                # 0
print(next(gen))                # 1
print(next(gen))                # 4

# 生成器只能用一次,再遍历空了
for x in 平方(3):
    print("for:", x)            # 0 1 4

**✏️ 练一练 ⭐⭐⭐**

写一个生成器 `fib(n)`,产出斐波那契数列的前 n 项(从 0, 1 开始)。

In [ ]:
# ======================
# 学员代码区
# ======================


In [ ]:
# 参考答案
def fib(n):
    a, b = 0, 1
    for _ in range(n):
        yield a
        a, b = b, a + b

print(list(fib(8)))             # [0, 1, 1, 2, 3, 5, 8, 13]

**3. 上下文管理器 `__enter__` / `__exit__`**

`with` 语句需要对象实现 `__enter__(进时自动执行)` 和 `__exit__(出时自动执行)`。`__exit__` 必须接受三个 `exc_` 参数。类比 Day07 
`with open` 自动关文件 —— 这个自动关的动作就是 `__exit__` 在做事。


In [ ]:
class Timer:
    def __enter__(self):
        import time
        self.start = time.time()
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        import time
        print(f"耗时 {time.time() - self.start:.4f}s")

# with 块退出时自动调 __exit__ 计时
with Timer():
    sum(range(1000000))

In [ ]:
# 参考答案
class ManagedFile:
    def __init__(self, path, mode):
        self.path = path
        self.mode = mode

    def __enter__(self):
        # 进入时打开文件并返回文件对象
        self.f = open(self.path, self.mode, encoding="utf-8")
        return self.f

    def __exit__(self, exc_type, exc_val, exc_tb):
        # 退出时关闭文件,自动释放资源
        self.f.close()

with ManagedFile("test.txt", "w") as f:
    f.write("hello")


In [ ]:
# ======================
# 学员代码区
# ======================

import functools

def retry(n):
    """失败时重试 n 次的装饰器"""
    def decorator(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            # TODO: 写重试逻辑
            pass
        return wrapper
    return decorator

@retry(3)
def risky():
    import random
    if random.random() < 0.7:
        raise ValueError("运气不好")
    return "成功!"

# risky()  # 试试看


**✏️ 练一练 ⭐⭐**

写一个上下文管理器 `ManagedFile(path, mode)`,进入时 `open` 文件并返回文件对象,退出时自动 `close`。用 `with ManagedFile("test.txt", "w") as f:` 写一行文字验证。

---



**4. 装饰器 `@timer`**

装饰器本质是**接收函数 → 返回新函数**的高阶函数,语法糖 `@decorator`。口诀:**装饰器 = 把函数装进壳,`@` 语法糖,`functools.wraps` 
留名**。永远写 `@functools.wraps(fn)`,否则被装饰的函数名会变成 `wrapper`,调试/文档全部出错。


In [ ]:
import functools, time

def timer(fn):
    @functools.wraps(fn)        # 保留原函数名/文档
    def wrapper(*args, **kwargs):
        start = time.time()
        result = fn(*args, **kwargs)
        print(f"{fn.__name__} 耗时 {time.time() - start:.4f}s")
        return result
    return wrapper

@timer                          # 等价于 slow = timer(slow)
def slow():
    time.sleep(0.1)

slow()                          # slow 耗时 0.1xxx s
print(slow.__name__)            # slow(被 wraps 保留)

**✏️ 练一练 ⭐⭐⭐⭐**

写一个带重试次数的装饰器 `@retry(n)`,失败时重试 n 次,每次打印重试信息。

In [ ]:
# ======================
# 学员代码区
# ======================


In [ ]:
# 参考答案
import functools

def retry(n):
    def decorator(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            for i in range(n + 1):
                try:
                    return fn(*args, **kwargs)
                except Exception as e:
                    print(f"第{i+1}次失败: {e}")
                    if i == n:
                        raise
        return wrapper
    return decorator

@retry(3)
def risky():
    import random
    if random.random() < 0.7:
        raise ValueError("运气不好")
    return "成功!"

# risky()  # 试试看

**5. 带参数的装饰器**

需要参数时再加一层:外层接收参数,中层接收函数,内层是真正包装。

In [ ]:
import functools

def log(level="INFO"):
    def decorator(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            print(f"[{level}] 调用 {fn.__name__}")
            return fn(*args, **kwargs)
        return wrapper
    return decorator

@log("DEBUG")
def add(a, b):
    return a + b

print(add(1, 2))                # [DEBUG] 调用 add → 3

**📝 本节试题集**

想更多练习?参见:
- 当堂练:`course/lesson09/in_class/practice01-06.py`(6 道)
- 课后作业:`course/lesson09/homework/task01-03.py`(3 道)

**今日小结**

import / from / as 三件套;`yield` 生成器惰性产出;`__enter__` / `__exit__` 自定义上下文管理器;`@decorator` + 
`@functools.wraps` 装饰器。

| 知识点 | 核心语法 | 一句话 |
|---|---|---|
| 导入 | `import` / `from` / `as` | 带前缀 / 直接用 / 别名 |
| 生成器 | `yield` | 产出不退场 |
| 上下文管理器 | `__enter__` / `__exit__` | 进出自动执行 |
| 装饰器 | `@decorator` + `@wraps` | 函数装进壳 |

> 🔴 **易错点**:装饰器漏 `@functools.wraps` 原函数名丢失;生成器遍历两次第二轮空;`__exit__` 签名忘加三个 `exc_` 参数报 
`TypeError`;`__init__.py` 必须保留。
